# Agentic AI — Hands-On Training Notebook

Welcome! This is **Section 1 & 2** of a 7-section hands-on training notebook covering Agentic AI concepts and tools.

### Before you run anything:
A `.env` file has been shared with you separately. Place it in the **same folder** as this notebook before running.
It contains: `HF_TOKEN`

You can generate a free HF_TOKEN at: https://huggingface.co/settings/tokens  
Select **'Read'** access — no payment required.

- Run cells **in order** from top to bottom

> **Note:** This notebook is designed to run as the first section of `training_notebook.ipynb`. Objects created here (`hf_client`, `embedding_model`) are used in later sections.

## ⚙️ Section 1: Setup & Environment

In [ ]:
# ✅ All packages are pre-installed in this Codespace
# This cell just verifies your environment is ready to go

from dotenv import load_dotenv
import os

# Load HF_TOKEN from Codespace secrets into environment
load_dotenv()

# Check the token is available
hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    print(f"✅ Environment ready!")
    print(f"   HF_TOKEN loaded: {hf_token[:6]}...")
    print(f"   Python environment: Codespace (all packages pre-installed)")
else:
    print("❌ HF_TOKEN not found.")
    print("   If running in Codespaces: check your secret is named HF_TOKEN exactly")
    print("   If running locally: add HF_TOKEN=your_token to a .env file")

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file in the same directory
load_dotenv()

# Retrieve the HuggingFace token from the environment (never print this!)
hf_token = os.getenv("HF_TOKEN")

# Guard: stop early with a clear message if the token is missing
if not hf_token:
    raise EnvironmentError(
        "HF_TOKEN not found. "
        "Make sure a .env file with HF_TOKEN=... is in the same folder."
    )

print(".env loaded — HF_TOKEN found (not shown for security)")

In [ ]:
from huggingface_hub import InferenceClient

# Initialise the HuggingFace Inference client using the loaded HF token
# Uses Qwen/Qwen2.5-72B-Instruct — a powerful open-weight instruction model
# This object will be reused throughout the training notebook
hf_client = InferenceClient(
    model="Qwen/Qwen2.5-72B-Instruct",
    token=os.environ.get("HF_TOKEN")
)

print("HuggingFace Inference client initialised — stored as 'hf_client'")
print("Model: Qwen/Qwen2.5-72B-Instruct")

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the sentence-transformers embedding model
# 'all-MiniLM-L6-v2' is a fast, lightweight model — great for demos and RAG pipelines
# This will download the model on first run (~80 MB), then cache it locally
print("Loading embedding model 'all-MiniLM-L6-v2' (may take a moment on first run)...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded — stored as 'embedding_model'")
print()
print("Environment ready!")

---

## 🔤 Section 2: Tokenisation

> **Run the cells below first — we'll explain what just happened after**

In [ ]:
from transformers import AutoTokenizer

# Load the tokenizer for Qwen2.5-7B from HuggingFace
# No API key needed — this only downloads the tokenizer config, not the full model
# Qwen2.5 uses Byte-Pair Encoding (BPE), the same algorithm used by most modern LLMs
print("Loading Qwen2.5-7B tokenizer from HuggingFace...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")
print("Tokenizer ready!")

In [ ]:
# --- Cell 1: Tokenise an English sentence ---

english_sentence = "The patient should take two tablets every morning with food."

# Encode the sentence into token IDs (the numbers the model actually sees)
english_token_ids = tokenizer.encode(english_sentence, add_special_tokens=False)

# Decode each individual token ID back to its string representation
english_tokens = [tokenizer.decode([tid]) for tid in english_token_ids]

print(f"Sentence  : {english_sentence}")
print(f"Token IDs : {english_token_ids}")
print(f"Tokens    : {english_tokens}")
print(f"Token count: {len(english_token_ids)}")

In [ ]:
# --- Cell 2: Tokenise the same sentence in French and compare ---

# Same meaning as the English sentence above, translated into French
french_sentence = "Le patient doit prendre deux comprimés chaque matin avec de la nourriture."

# Encode the French sentence
french_token_ids = tokenizer.encode(french_sentence, add_special_tokens=False)

# Decode each token back to its string
french_tokens = [tokenizer.decode([tid]) for tid in french_token_ids]

print(f"Sentence (FR) : {french_sentence}")
print(f"Token IDs     : {french_token_ids}")
print(f"Tokens        : {french_tokens}")
print(f"Token count   : {len(french_token_ids)}")
print()
print("--- Comparison ---")
print(f"English token count : {len(english_token_ids)}")
print(f"French token count  : {len(french_token_ids)}")
print(f"Difference          : {len(french_token_ids) - len(english_token_ids)} more tokens in French")
print()
print("Observation: The same meaning in French uses more tokens.")
print("This is because most LLM tokenizers are trained on English-heavy data,")
print("so non-English words are split into more sub-word pieces.")

In [ ]:
# --- Cell 3: Context-dependent tokenisation — the word 'bank' ---

# 'bank' in a financial context
sentence_a = "I need to go to the bank to deposit my cheque."

# 'bank' in a geographical/nature context
sentence_b = "The fisherman sat on the river bank all afternoon."

# Tokenise both sentences
tokens_a = [tokenizer.decode([tid]) for tid in tokenizer.encode(sentence_a, add_special_tokens=False)]
tokens_b = [tokenizer.decode([tid]) for tid in tokenizer.encode(sentence_b, add_special_tokens=False)]

print(f"Sentence A: {sentence_a}")
print(f"Tokens A  : {tokens_a}")
print(f"Count A   : {len(tokens_a)}")
print()
print(f"Sentence B: {sentence_b}")
print(f"Tokens B  : {tokens_b}")
print(f"Count B   : {len(tokens_b)}")
print()
# Find where 'bank' appears in the token list for each sentence
bank_tokens_a = [t for t in tokens_a if 'bank' in t.lower()]
bank_tokens_b = [t for t in tokens_b if 'bank' in t.lower()]
print(f"'bank' token(s) in sentence A: {bank_tokens_a}")
print(f"'bank' token(s) in sentence B: {bank_tokens_b}")
print()
print("Note: The token for 'bank' is the same in both sentences.")
print("The model understands context NOT through the token itself,")
print("but through the surrounding tokens and its attention mechanism.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import math

def plot_token_grid(tokens, title="Token Visualisation", figsize=(12, 4)):
    """
    Draw a coloured grid where each box is one token.
    Tokens wrap onto new rows when a row fills up.
    """
    # Pick a set of visually distinct colours — cycling through them for each token
    colour_palette = list(mcolors.TABLEAU_COLORS.values())  # 10 distinct colours

    # How many tokens to show per row before wrapping
    tokens_per_row = 10
    n_rows = math.ceil(len(tokens) / tokens_per_row)

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, tokens_per_row)
    ax.set_ylim(0, n_rows)
    ax.axis("off")  # Hide axis ticks and labels
    ax.set_title(title, fontsize=13, fontweight="bold", pad=12)

    for i, token in enumerate(tokens):
        # Calculate grid position: column and row (rows grow downward)
        col = i % tokens_per_row
        row = n_rows - 1 - (i // tokens_per_row)  # flip so row 0 is at top

        # Assign a colour based on token index, cycling through the palette
        colour = colour_palette[i % len(colour_palette)]

        # Draw the coloured rectangle for this token
        rect = mpatches.FancyBboxPatch(
            (col + 0.05, row + 0.15),  # (x, y) position with a small margin
            0.88, 0.70,                # width, height
            boxstyle="round,pad=0.02",
            facecolor=colour,
            edgecolor="white",
            linewidth=1.5,
            alpha=0.85
        )
        ax.add_patch(rect)

        # Write the token string label inside the box
        # Use repr() to make whitespace tokens like ' the' clearly visible
        label = repr(token).strip("'")
        ax.text(
            col + 0.49, row + 0.50,  # centre of the box
            label,
            ha="center", va="center",
            fontsize=8, color="white", fontweight="bold"
        )

    plt.tight_layout()
    plt.show()


# Visualise the English sentence tokens
plot_token_grid(
    english_tokens,
    title=f"Tokens: '{english_sentence}' ({len(english_tokens)} tokens)"
)

# Visualise the French sentence tokens for comparison
plot_token_grid(
    french_tokens,
    title=f"Tokens: '{french_sentence}' ({len(french_tokens)} tokens)"
)

### What just happened?

#### What are tokens?

Large language models (LLMs) do not read text the way humans do — character by character or word by word. Instead, they operate on **tokens**: sub-word chunks produced by a tokenizer algorithm (typically **Byte-Pair Encoding, BPE**). Common English words are usually a single token; rare words, names, or non-English text get split into multiple sub-word pieces.

For example:
- `"tablet"` → 1 token
- `"immunoglobulin"` → likely 3–4 tokens
- `"comprimés"` (French for tablets) → 2–3 tokens

#### Why does it matter?

| Implication | Detail |
|---|---|
| **Cost** | LLM APIs charge per token. Verbose prompts or non-English content cost more. |
| **Context window** | Every model has a maximum token limit. Long documents must be chunked. |
| **Non-English languages** | Most tokenizers are English-biased — the same meaning in French or Chinese uses more tokens. |
| **Prompt engineering** | Knowing roughly how many tokens your prompt uses helps you stay within limits and control cost. |
| **Model reasoning** | The model predicts one token at a time — understanding this helps you write clearer, less ambiguous prompts. |

#### Rule of thumb
In English, **1 token ≈ 0.75 words**, or roughly **100 tokens ≈ 75 words**. This varies significantly for other languages and specialised vocabulary (e.g. chemical names, code).

In [ ]:
# --- Try it yourself! ---
# Change the sentence below and re-run this cell to see how your text is tokenised.

my_sentence = "Type your own sentence here and see how it gets broken into tokens!"

# Tokenise the custom sentence
my_token_ids = tokenizer.encode(my_sentence, add_special_tokens=False)
my_tokens = [tokenizer.decode([tid]) for tid in my_token_ids]

print(f"Your sentence : {my_sentence}")
print(f"Tokens        : {my_tokens}")
print(f"Token count   : {len(my_token_ids)}")

# Visualise your tokens as a coloured grid
plot_token_grid(
    my_tokens,
    title=f"Your tokens ({len(my_tokens)} total)"
)

---

> **Note:** This notebook is designed to run after the previous sections in `training_notebook.ipynb`.
> It assumes `hf_client` (HuggingFace InferenceClient for Qwen2.5-72B-Instruct) is already initialised in memory.

> **API Key:** A `.env` file has been shared with you separately. Place it in the same folder as this notebook before running. It contains: `HF_TOKEN`
> You can generate a free HF_TOKEN at: https://huggingface.co/settings/tokens — select **Read** access, no payment required.

## 💬 Section 3: Prompt Engineering

The same question. Five different ways to ask it. Very different answers.

We'll look at:
- **Zero-Shot** — just ask
- **Few-Shot** — show examples first
- **Chain-of-Thought** — ask the model to reason aloud
- **Role-Based** — give the model an identity
- **RAG-Based** — hand the model relevant context before asking

---

**Run the cells below first — we'll explain what just happened after.**

In [ ]:
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
import os

# Load HF_TOKEN from the .env file in the same folder as this notebook
load_dotenv()

# Initialise the HuggingFace Inference client for Qwen2.5-72B-Instruct
hf_client = InferenceClient(
    model="Qwen/Qwen2.5-72B-Instruct",
    token=os.environ.get("HF_TOKEN")   # never hardcoded — always from .env
)

# ── Shared question used across ALL five prompt types ──────────────────────────
QUESTION = (
    "A batch record shows a temperature excursion of 2°C above the validated range "
    "for 45 minutes. What should happen next?"
)

# Helper: send a prompt to HuggingFace / Qwen2.5 and return the text response
def ask_hf(prompt: str) -> str:
    """Send a single-turn prompt to Qwen2.5-72B-Instruct and return the assistant reply."""
    response = hf_client.chat.completions.create(
        model="Qwen/Qwen2.5-72B-Instruct",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512,
        temperature=0.7
    )
    return response.choices[0].message.content

# Helper: pretty-print each result with a consistent layout
def show_result(label: str, prompt: str, response: str) -> None:
    """Print the prompt type label, the full prompt, a divider, then the response."""
    print(f"\n{'='*70}")
    print(f"  PROMPT TYPE: {label}")
    print(f"{'='*70}")
    print("--- FULL PROMPT SENT ---")
    print(prompt)
    print("\n--- MODEL RESPONSE ---")
    print(response)
    print(f"{'='*70}\n")

print("HuggingFace client initialised. Shared question loaded and helper functions defined.")
print(f"Question: {QUESTION}")

---
### 1 of 5 — Zero-Shot
Ask the question with no examples, no persona, no extra context. The model relies entirely on what it learned during pre-training.

In [ ]:
# ── ZERO-SHOT ──────────────────────────────────────────────────────────────────
# No examples, no persona, no added context — just the bare question.
# This is the baseline: fastest to write, least control over output.

zero_shot_prompt = QUESTION  # the question is the entire prompt

zero_shot_response = ask_hf(zero_shot_prompt)  # call HuggingFace / Qwen2.5

show_result("Zero-Shot", zero_shot_prompt, zero_shot_response)

---
### 2 of 5 — Few-Shot
Provide two worked examples *before* the real question. The model picks up the pattern — format, tone, depth — and mirrors it in its answer.

In [ ]:
# ── FEW-SHOT ───────────────────────────────────────────────────────────────────
# Two Q&A examples prime the model on the expected response style before we ask
# the real question. The model will mirror the format and depth of the examples.

few_shot_prompt = """
You are answering pharmaceutical manufacturing compliance questions.
Here are two example questions and the kind of answer expected:

Q: A label printer malfunctions mid-batch and prints incomplete labels on 20 units.
   What should happen next?
A: Immediately quarantine the 20 affected units. Raise a deviation report documenting
   the equipment failure, the batch number, and the quantity affected. QA must assess
   whether the units can be re-labelled under a controlled procedure or must be
   destroyed. The printer must be taken out of service pending investigation and
   re-qualification before returning to production.

Q: An operator notices that the cleaning log for a reactor vessel is missing a
   sign-off for the previous batch cycle. What should happen next?
A: Stop use of the vessel immediately and place it under quarantine. Raise a
   deviation and initiate a retrospective investigation to determine whether cleaning
   was actually performed. If cleaning cannot be verified, the vessel must be
   re-cleaned and re-qualified. All batches manufactured in the intervening period
   may require impact assessment.

Now answer the following question in the same style:
Q: {question}
A:
""".format(question=QUESTION)  # inject the shared question into the template

few_shot_response = ask_hf(few_shot_prompt)  # call HuggingFace / Qwen2.5

show_result("Few-Shot", few_shot_prompt, few_shot_response)

---
### 3 of 5 — Chain-of-Thought
Asking the model to *think step by step* forces it to externalise its reasoning. This often produces more accurate, auditable answers — especially for multi-step compliance decisions.

In [ ]:
# ── CHAIN-OF-THOUGHT ───────────────────────────────────────────────────────────
# Instructing the model to "think step by step" encourages it to break the problem
# into sequential reasoning stages before arriving at a conclusion.
# This reduces errors on multi-step problems and makes the logic auditable.

cot_prompt = """
You are a pharmaceutical manufacturing expert. Think step by step before giving
your final recommendation.

Question: {question}

Walk through your reasoning step by step, then state your final recommendation.
""".format(question=QUESTION)  # inject the shared question

cot_response = ask_hf(cot_prompt)  # call HuggingFace / Qwen2.5

show_result("Chain-of-Thought", cot_prompt, cot_response)

---
### 4 of 5 — Role-Based
Assigning a specific identity changes the model's "perspective": vocabulary, assumed knowledge, and risk tolerance all shift to match the role. This is one of the cheapest ways to steer output quality.

In [ ]:
# ── ROLE-BASED ─────────────────────────────────────────────────────────────────
# Giving the model a detailed persona — company, title, expertise — anchors its
# vocabulary and reasoning to that domain. Notice how the tone and specificity
# of the response shifts compared to Zero-Shot.

role_prompt = """
You are a Senior Pharmaceutical Compliance Expert at Eli Lilly with 15 years of
experience in GMP manufacturing environments. You are advising a QA Manager.
Your answers must reference regulatory expectations (e.g. FDA 21 CFR Part 211,
EU GMP Annex 15) where relevant, and always consider patient safety first.

Question: {question}
""".format(question=QUESTION)  # inject the shared question

role_response = ask_hf(role_prompt)  # call HuggingFace / Qwen2.5

show_result("Role-Based", role_prompt, role_response)

---
### 5 of 5 — RAG-Based (Retrieval-Augmented Generation)
Instead of relying on the model's pre-trained knowledge, we inject a relevant excerpt from a source document directly into the prompt. The model is explicitly told to use *that* text — grounding the answer in your own documentation.

In [ ]:
# ── RAG-BASED ──────────────────────────────────────────────────────────────────
# We simulate retrieval by hard-coding a relevant SOP excerpt as the context.
# In a real RAG pipeline (see nb_03_rag_pipeline.ipynb) this excerpt would be
# fetched dynamically from a vector store based on the question.

# --- Simulated retrieved context from internal documentation ---
retrieved_context = """
--- Retrieved from: SOP-QA-0042 Rev 3 — Temperature Excursion Management ---

3.1 IMMEDIATE ACTIONS
Upon identification of any temperature excursion from the validated storage or
processing range, the operator must:
  a) Record the exact time, duration, and magnitude of the excursion in the batch record.
  b) Notify the QA designee and batch record owner within 30 minutes of detection.
  c) Place the affected batch on HOLD status in the ERP system pending assessment.

3.2 IMPACT ASSESSMENT
QA and the process owner must jointly complete Form QA-0042-F1 (Excursion Impact
Assessment) within 24 hours. The assessment must consider:
  - Product stability data relevant to the excursion conditions
  - Duration and magnitude relative to validated limits
  - Whether the excursion was within any pre-approved design space

3.3 DISPOSITION
Batch disposition (release, retest, or reject) must be approved by QA Director
and documented in the batch record. Regulatory reporting obligations must be
evaluated if the product is already in distribution.
"""

# Build the RAG prompt: context block + question + explicit grounding instruction
rag_prompt = """
Use ONLY the following internal SOP excerpt to answer the question below.
If the excerpt does not contain sufficient information, say so explicitly —
do not supplement with outside knowledge.

{context}

Question: {question}
""".format(context=retrieved_context, question=QUESTION)  # inject both

rag_response = ask_hf(rag_prompt)  # call HuggingFace / Qwen2.5

show_result("RAG-Based", rag_prompt, rag_response)

---
### At-a-Glance Comparison

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Comparison table data ──────────────────────────────────────────────────────
# Each row: [Prompt Type, Effort to Write, Control Over Output, Best Use Case]
table_data = [
    ["Zero-Shot",       "Very Low",  "Low",       "Quick exploration, prototyping"],
    ["Few-Shot",        "Medium",    "Medium",    "Consistent format / tone across many queries"],
    ["Chain-of-Thought","Low",       "Medium",    "Complex multi-step reasoning & decisions"],
    ["Role-Based",      "Low",       "Medium-High","Domain-specific tone, regulatory language"],
    ["RAG-Based",       "High",      "High",      "Grounding answers in verified source documents"],
]

col_labels = ["Prompt Type", "Effort to Write", "Control Over Output", "Best Use Case"]

# ── Figure setup ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis("off")  # hide the axes — we only want the table

# ── Draw table ─────────────────────────────────────────────────────────────────
table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc="center",
    loc="center",
    colWidths=[0.18, 0.16, 0.18, 0.48],  # proportion of total width per column
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.2)  # stretch rows vertically for readability

LILLY_RED  = "#C8102E"   # Eli Lilly brand red
LILLY_LIGHT = "#F9E8EB"  # very light tint for alternating rows
WHITE       = "#FFFFFF"

# Style the header row (row index 0)
for col_idx in range(len(col_labels)):
    cell = table[0, col_idx]           # row 0 = header
    cell.set_facecolor(LILLY_RED)      # Lilly red background
    cell.set_text_props(color="white", fontweight="bold")  # white bold text
    cell.set_edgecolor("white")

# Style data rows with alternating light tint
for row_idx in range(1, len(table_data) + 1):  # rows 1–5
    bg = LILLY_LIGHT if row_idx % 2 == 0 else WHITE  # alternate shading
    for col_idx in range(len(col_labels)):
        cell = table[row_idx, col_idx]
        cell.set_facecolor(bg)
        cell.set_edgecolor("#DDDDDD")  # subtle grey borders
        # Left-align the last column (Best Use Case) for readability
        if col_idx == 3:
            cell.get_text().set_ha("left")

ax.set_title(
    "Prompt Engineering Strategies — At a Glance",
    fontsize=14,
    fontweight="bold",
    color=LILLY_RED,
    pad=20,
)

plt.tight_layout()
plt.show()  # render the figure inline

---
## 📖 What Just Happened — And Why It Matters

You ran the same question through five fundamentally different prompting strategies. Here's how to think about the tradeoffs:

---

### Zero-Shot
**What it is:** Ask the question with nothing else attached.  
**Tradeoff:** Lowest friction to write, but the model has to infer everything — domain, format, depth, risk tolerance. Results are often usable but generic.  
**Use when:** Exploring whether a model understands a topic at all, or when you need a quick first draft.

---

### Few-Shot
**What it is:** Show the model two or three worked examples before the real question.  
**Tradeoff:** The model mirrors the format and depth of your examples — so good examples produce good outputs, but bad examples are faithfully reproduced too.  
**Use when:** You need consistent structure across many queries (e.g. batch deviation summaries should always follow the same template).

---

### Chain-of-Thought (CoT)
**What it is:** Instruct the model to reason step by step before answering.  
**Tradeoff:** Improves accuracy on multi-step problems by forcing the model to externalise intermediate logic. The answer is longer, but the reasoning is auditable. Can still hallucinate — it just does so transparently.  
**Use when:** The question requires sequential reasoning: root cause analysis, risk assessment, eligibility determinations.

---

### Role-Based
**What it is:** Give the model an explicit identity — job title, company, experience level, regulatory remit.  
**Tradeoff:** Very cheap to write (one sentence), surprisingly effective at shifting vocabulary, assumed expertise, and risk framing. However, the model is still working from pre-trained knowledge — a "Senior GMP Expert" persona does not grant access to your actual SOPs.  
**Use when:** You need domain-appropriate tone and terminology: regulatory writing, clinical summaries, audit-ready language.

---

### RAG-Based (Retrieval-Augmented Generation)
**What it is:** Inject a relevant excerpt from a verified source document into the prompt and instruct the model to answer *from that text only*.  
**Tradeoff:** Highest setup effort (you need a retrieval mechanism), but the answer is grounded in *your* documentation rather than the model's pre-trained weights. Hallucination risk drops significantly. If the context doesn't contain the answer, a well-prompted RAG response will say so.  
**Use when:** Accuracy and traceability matter — answering from SOPs, regulatory guidelines, validated data. This is the foundation of the RAG pipeline you'll build in Section 4.

---

### The Core Principle

> **A language model can only work with what you give it.**  
> Prompt choice is the primary lever you have over output quality *before* any fine-tuning or RAG infrastructure is in place.

In practice, the most effective prompts **combine** strategies: a role-based persona + chain-of-thought reasoning + RAG context is a common production pattern. You'll see this in action in Section 5 (Agentic AI).

---
## 🧪 Try It Yourself

Change the `question` variable below and re-run the cell to compare all five prompt strategies on your own question.

In [ ]:
# ── TRY IT YOURSELF ────────────────────────────────────────────────────────────
# Swap out the question below, then run this cell.
# All five prompt types will be applied automatically.

question = "A batch record shows a temperature excursion of 2°C above the validated range for 45 minutes. What should happen next?"
# ↑ Replace the text above with any question you like, then run the cell.

# ── Small hardcoded context for the RAG variant ────────────────────────────────
# In a real pipeline this would be retrieved from a vector store.
# Update this string if your question is about a different topic.
context = """
--- Retrieved from: SOP-QA-0042 Rev 3 — Temperature Excursion Management ---

3.1 IMMEDIATE ACTIONS
Upon identification of any temperature excursion from the validated storage or
processing range, the operator must:
  a) Record the exact time, duration, and magnitude of the excursion in the batch record.
  b) Notify the QA designee and batch record owner within 30 minutes of detection.
  c) Place the affected batch on HOLD status in the ERP system pending assessment.

3.2 IMPACT ASSESSMENT
QA and the process owner must jointly complete Form QA-0042-F1 (Excursion Impact
Assessment) within 24 hours.

3.3 DISPOSITION
Batch disposition (release, retest, or reject) must be approved by QA Director.
"""

# ── Build and run all five prompts ─────────────────────────────────────────────

# 1. Zero-Shot — bare question, no extras
prompts = {
    "Zero-Shot": question,

    # 2. Few-Shot — two worked examples before the question
    "Few-Shot": (
        "You are answering pharmaceutical manufacturing compliance questions.\n"
        "Here are two example questions and the kind of answer expected:\n\n"
        "Q: A label printer malfunctions mid-batch. What should happen next?\n"
        "A: Quarantine affected units, raise a deviation, assess for re-labelling or destroy.\n\n"
        "Q: A cleaning log sign-off is missing. What should happen next?\n"
        "A: Quarantine the vessel, raise a deviation, verify or re-clean, assess impacted batches.\n\n"
        f"Now answer:\nQ: {question}\nA:"
    ),

    # 3. Chain-of-Thought — step-by-step reasoning before the conclusion
    "Chain-of-Thought": (
        "You are a pharmaceutical manufacturing expert. Think step by step before giving "
        "your final recommendation.\n\n"
        f"Question: {question}\n\n"
        "Walk through your reasoning step by step, then state your final recommendation."
    ),

    # 4. Role-Based — explicit persona with regulatory expertise
    "Role-Based": (
        "You are a Senior Pharmaceutical Compliance Expert at Eli Lilly with 15 years of "
        "experience in GMP manufacturing. You are advising a QA Manager. Reference relevant "
        "regulations (FDA 21 CFR Part 211, EU GMP) where applicable.\n\n"
        f"Question: {question}"
    ),

    # 5. RAG-Based — grounded in the provided context excerpt
    "RAG-Based": (
        "Use ONLY the following internal SOP excerpt to answer the question. "
        "If the excerpt does not contain sufficient information, say so explicitly.\n\n"
        f"{context}\n\nQuestion: {question}"
    ),
}

# Iterate through all five and display results
for prompt_type, prompt_text in prompts.items():
    print(f"\nRunning: {prompt_type} ...")      # progress indicator
    response = ask_hf(prompt_text)              # call HuggingFace / Qwen2.5
    show_result(prompt_type, prompt_text, response)  # display formatted output

---

## 📚 Section 4: RAG Pipeline

> **Note:** This notebook is designed to run after the previous sections in `training_notebook.ipynb`.
>
> When merged and run top-to-bottom, the following object is already in memory from an earlier section:
> - `hf_client` — HuggingFace Inference client instance *(created in Section 1)*
>
> This section **creates** `embedding_model` and `chroma_collection`, which later notebooks will reuse.

*Run the cells below first — we'll explain what just happened after.*

In [ ]:
# Install packages required for this section
# sentence-transformers: local embedding model
# chromadb: in-memory vector store
# matplotlib: pipeline diagram
%pip install sentence-transformers chromadb matplotlib --index-url https://pypi.org

In [ ]:
# ── Cell 1: Load & Chunk ──────────────────────────────────────────────────────
# Six fictional Lilly SOP documents — realistic-sounding, completely fabricated

documents = [

    # SOP 1 — Temperature excursion handling
    """SOP-QA-0042 | Temperature Excursion Handling Procedure
Scope: All temperature-controlled storage areas in Manufacturing and QC.
1. Upon detection of an excursion, the operator must log the event in the EMS
   (Environmental Monitoring System) within 15 minutes of discovery.
2. Quarantine all affected materials by applying a HOLD label (form QA-F-0019).
3. Notify the QA On-Call within 30 minutes using the site escalation matrix.
4. Complete a Temperature Excursion Assessment Form (QA-F-0042A) documenting
   duration, extent, and affected batch numbers.
5. QA determines disposition: release, additional testing, or destruction.
Responsible parties: Manufacturing Operator, QA On-Call, Site Quality Director.""",

    # SOP 2 — Deviation reporting and classification
    """SOP-QA-0017 | Deviation Reporting and Classification
Scope: All GMP-regulated activities across Manufacturing, QC, and Supply Chain.
1. Any departure from an approved procedure or specification is a deviation.
2. Deviations must be initiated in TrackWise within 24 hours of discovery.
3. Classification: Minor (no quality impact), Major (potential quality impact),
   Critical (confirmed or probable patient safety risk).
4. Major and Critical deviations require immediate QA notification and batch quarantine.
5. Root cause analysis: 30 calendar days for Major, 15 calendar days for Critical.
6. All deviations require QA approval prior to batch release.
Responsible parties: Initiating Department, QA Reviewer, Site Quality Director.""",

    # SOP 3 — CAPA initiation and documentation
    """SOP-QA-0055 | CAPA Initiation and Documentation
Scope: Corrective and Preventive Action process for all GMP sites.
1. CAPAs are initiated from deviations, audit findings, complaints, or trend data.
2. A CAPA owner must be assigned within 5 business days of the triggering event.
3. The CAPA plan must include: problem statement, root cause method (5-Why, Fishbone,
   or FMEA), corrective actions, preventive actions, and effectiveness criteria.
4. Effectiveness checks must be scheduled no later than 90 days post-implementation.
5. Supporting evidence (training records, updated SOPs, logs) must be attached.
6. CAPAs cannot be closed without documented evidence of effectiveness.
Responsible parties: CAPA Owner, QA Approver, Regulatory Affairs (if PAS required).""",

    # SOP 4 — Batch record review
    """SOP-QA-0031 | Batch Record Review Process
Scope: Review and approval of executed batch records for all commercial drug products.
1. Completed batch records must be submitted to QA Review within 3 business days of
   batch completion or last in-process test result, whichever is later.
2. QA reviewers verify: completeness, legibility, correction procedure compliance
   (single line through error, initials, date, reason), yield calculations, in-process results.
3. Missing or illegible entries must be clarified via a formal query to Manufacturing.
4. Sterile products require dual QA review.
5. Approved records are retained in the Document Management System for 30 years
   or 1 year post-expiry of the last batch, whichever is longer.
Responsible parties: QA Reviewer, QA Approver, Manufacturing Supervisor.""",

    # SOP 5 — Out-of-specification investigation
    """SOP-QA-0028 | Out-of-Specification (OOS) Investigation Steps
Scope: Laboratory investigations for any result outside established acceptance criteria.
Phase 1 — Laboratory Investigation (complete within 5 business days):
1. Review raw data, instrument logs, and analyst records for errors or miscalculations.
2. Assess reagent, reference standard, column, and instrument suitability.
3. If an assignable lab cause is found and documented, the result may be invalidated;
   re-testing on the original sample is permitted.
Phase 2 — Full Investigation (if Phase 1 inconclusive, complete within 20 business days):
1. Expand to manufacturing process, raw materials, and environmental conditions.
2. Retain all samples; do not dispose until investigation is closed.
3. Statistical retesting requires prior QA and Regulatory approval.
Final disposition requires QA and Site Quality Director sign-off.
Responsible parties: Analyst, Lab Supervisor, QA OOS Coordinator.""",

    # SOP 6 — Change control
    """SOP-QA-0009 | Change Control Procedure Overview
Scope: All changes to processes, equipment, facilities, materials, and documentation.
1. A Change Control Request (CCR) must be submitted in TrackWise before implementation.
2. The CCR must include: change description, reason, impact assessment (quality,
   regulatory, validation), and implementation plan.
3. Cross-functional review required for Major changes (QA, Regulatory, Manufacturing,
   Validation).
4. Regulatory Affairs assesses whether a Prior Approval Supplement, CBE-30, or Annual
   Report filing is required based on product registration commitments.
5. All validation or qualification activities must be completed before go-live.
6. Post-implementation review must occur within 60 days to verify effectiveness.
Responsible parties: Change Owner, QA Approver, Regulatory Affairs, Validation Lead."""
]

# Print each document with its index for inspection
for i, doc in enumerate(documents):
    print(f"── Document {i} ──────────────────────────────────────────")
    print(doc[:220], "...")    # first 220 chars keeps output readable
    print()

In [ ]:
# ── Cell 2: Embed & Store ─────────────────────────────────────────────────────
import chromadb                                            # in-memory vector database
from sentence_transformers import SentenceTransformer      # local embedding model

# Initialise the embedding model — produces 384-dimensional sentence vectors
# This object (embedding_model) will be reused by later notebooks
print("Loading embedding model (all-MiniLM-L6-v2)...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded.\n")

# Create an ephemeral ChromaDB client — everything lives in RAM, no disk writes
chroma_client = chromadb.Client()

# Create the named collection — this object (chroma_collection) will be reused later
chroma_collection = chroma_client.create_collection(
    name="lilly_sops",
    metadata={"hnsw:space": "cosine"}    # use cosine similarity for retrieval
)

# Embed all documents in a single batch call — returns a numpy array of shape (6, 384)
print("Embedding documents...")
embeddings = embedding_model.encode(documents, show_progress_bar=True)

# Build unique IDs and metadata for each chunk
ids       = [f"sop_{i}" for i in range(len(documents))]              # e.g. "sop_0", "sop_1"
metadatas = [{"source": f"SOP document {i}"} for i in range(len(documents))]

# Add documents, vectors, and metadata to the collection in one call
chroma_collection.add(
    ids=ids,
    documents=documents,                  # raw text stored alongside vectors for retrieval
    embeddings=embeddings.tolist(),        # ChromaDB expects plain Python lists, not numpy
    metadatas=metadatas
)

# Confirm how many chunks are now in the store
print(f"\nStored {chroma_collection.count()} documents in ChromaDB")

In [ ]:
# ── Cell 3: Retrieve ──────────────────────────────────────────────────────────
# Embed a user query and find the most semantically similar SOP chunks

sample_query = "What should I do if a batch result comes back outside specification?"

print(f"Query: {sample_query}\n")

# Embed the query — MUST use the same model as the documents (embedding space must match)
query_embedding = embedding_model.encode([sample_query]).tolist()   # shape becomes (1, 384)

# Ask ChromaDB for the top-2 closest chunks by cosine similarity
results = chroma_collection.query(
    query_embeddings=query_embedding,
    n_results=2,                                          # retrieve 2 chunks
    include=["documents", "distances", "metadatas"]       # what to return per hit
)

# Unpack results — ChromaDB wraps everything in an outer list (one entry per query)
retrieved_docs = results["documents"][0]     # list of document strings
distances      = results["distances"][0]     # cosine distances (0 = identical, 2 = opposite)

print("── Top Retrieved Chunks ──────────────────────────────────────")
for rank, (doc, dist) in enumerate(zip(retrieved_docs, distances), start=1):
    similarity = 1 - dist                    # convert cosine distance → similarity score
    print(f"\nRank {rank}  |  Similarity Score: {similarity:.4f}")
    print(f"Source: {results['metadatas'][0][rank - 1]['source']}")
    print("-" * 60)
    print(doc[:420], "...")                  # first 420 chars of each retrieved chunk
    print()

In [ ]:
# ── Cell 4: Generate ──────────────────────────────────────────────────────────
# Inject retrieved chunks as grounded context into the HuggingFace / Qwen2.5 prompt

# Concatenate the top-2 chunks into a single context block
context = "\n\n---\n\n".join(retrieved_docs)     # separator makes chunk boundaries visible

# RAG prompt pattern: system instruction + retrieved context + user question
rag_prompt = f"""You are a helpful pharmaceutical quality assistant.
Use ONLY the context provided below to answer the question.
If the answer is not found in the context, say "I cannot find that in the provided SOPs."

CONTEXT:
{context}

QUESTION: {sample_query}

ANSWER:"""

print("Sending grounded prompt to HuggingFace / Qwen2.5...\n")

# hf_client is already initialised — created in nb_01 (Section 1)
response = hf_client.chat.completions.create(
    model="Qwen/Qwen2.5-72B-Instruct",     # HuggingFace hosted Qwen2.5 model
    messages=[
        {"role": "user", "content": rag_prompt}    # single-turn RAG call
    ],
    max_tokens=512,
    temperature=0.7
)

# Extract the assistant's reply from the response object
answer = response.choices[0].message.content

print("── Grounded Answer ───────────────────────────────────────────")
print(answer)

In [ ]:
# ── RAG Pipeline Flowchart ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 4))
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.axis("off")                     # hide axes — we draw everything manually

# Lilly red for box borders; light blush for fill
LILLY_RED  = "#C8102E"
BOX_FILL   = "#FFF0F0"
TEXT_COLOR = "#1A1A1A"

# Each box: (x_left, y_bottom, width, height, label)
boxes = [
    (0.30,  1.1, 2.3, 1.8, "Load &\nChunk"),
    (3.05,  1.1, 2.3, 1.8, "Embed"),
    (5.80,  1.1, 2.3, 1.8, "Store"),
    (8.55,  1.1, 2.3, 1.8, "Retrieve"),
    (11.30, 1.1, 2.3, 1.8, "Generate"),
]

# Draw each rounded rectangle
for (x, y, w, h, label) in boxes:
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.12",
        linewidth=2.5,
        edgecolor=LILLY_RED,
        facecolor=BOX_FILL
    )
    ax.add_patch(patch)                         # add box to the axes
    ax.text(                                    # centred label inside the box
        x + w / 2, y + h / 2, label,
        ha="center", va="center",
        fontsize=12, fontweight="bold",
        color=TEXT_COLOR
    )

# Draw arrows connecting consecutive boxes at their mid-height
for i in range(len(boxes) - 1):
    x_tail = boxes[i][0] + boxes[i][2]           # right edge of current box
    x_head = boxes[i + 1][0]                     # left edge of next box
    y_mid  = boxes[i][1] + boxes[i][3] / 2       # vertical centre of box

    ax.annotate(
        "",
        xy=(x_head, y_mid),                      # arrow tip
        xytext=(x_tail, y_mid),                  # arrow tail
        arrowprops=dict(
            arrowstyle="->",
            color=LILLY_RED,
            lw=2.5,
            mutation_scale=22                    # controls arrowhead size
        )
    )

ax.set_title(
    "RAG Pipeline — Retrieval-Augmented Generation",
    fontsize=15, fontweight="bold", color=LILLY_RED, pad=12
)
plt.tight_layout()
plt.show()

## 🧠 Why RAG? Theory Behind the Approach

### The Core Idea
Instead of asking an LLM to **recall** facts from its training data (unreliable, frozen in time), RAG **retrieves** the relevant passage first and asks the model to **read and summarise** it. The model is doing comprehension, not recall — a far more reliable task.

### RAG vs Fine-Tuning — When to Use Each

| | Fine-Tuning | RAG |
|---|---|---|
| **Best for** | Teaching new reasoning patterns or writing style | Grounding answers in specific, retrievable facts |
| **Data freshness** | Model must be fully retrained on every update | Add/update documents in the vector store — no retraining needed |
| **Private data** | Knowledge is baked into model weights (data leaves your control) | Documents stay in your own infrastructure |
| **Hallucination risk** | Higher — model "remembers" but can confabulate | Lower — answer is anchored to retrieved text |
| **Cost** | High — GPU compute for training runs | Low — embedding + retrieval + one LLM call |
| **Auditability** | Hard to trace which training example drove an answer | You can cite the exact source chunk |

### How RAG Reduces Hallucination
The prompt explicitly instructs the model to use **only** the provided context. This shifts the task from open-ended generation to **closed-book reading comprehension**. If the answer is not in the retrieved chunks, a well-prompted model will say so rather than fabricate.

### When to Choose RAG
- **Frequently updated data** — SOPs revised quarterly, drug labels changing, regulations evolving. Fine-tuned models go stale; RAG doesn't.
- **Private / confidential data** — clinical documents, trade secrets. Documents never need to leave your infrastructure.
- **Auditability requirements** — pharmaceutical QA, legal, finance. You can point to the exact source chunk that drove the answer.
- **Factual precision at scale** — when hallucination is unacceptable, ground every answer in retrieved evidence.

> **Rule of thumb:** If your knowledge base changes more than once a month, or if the data is confidential, RAG is almost always the better choice over fine-tuning.

In [ ]:
# ── Try It Yourself ───────────────────────────────────────────────────────────
# Change user_question below and re-run this cell to see different retrievals and answers

user_question = "How do I initiate a CAPA and what documentation is required?"  # ← edit me!

# ── Step 1: Embed the user's question ────────────────────────────────────────
print(f"Your question: {user_question}\n")
user_query_vec = embedding_model.encode([user_question]).tolist()   # embed using same model

# ── Step 2: Retrieve the top-2 most relevant SOP chunks ───────────────────────
user_hits = chroma_collection.query(
    query_embeddings=user_query_vec,
    n_results=2,                                           # fetch 2 closest chunks
    include=["documents", "distances", "metadatas"]
)

user_top_docs  = user_hits["documents"][0]                 # retrieved document strings
user_distances = user_hits["distances"][0]                 # cosine distances

print("── Retrieved Chunks ──────────────────────────────────────────")
for rank, (doc, dist) in enumerate(zip(user_top_docs, user_distances), start=1):
    print(f"\nRank {rank}  |  Similarity: {1 - dist:.4f}")
    print(f"Source: {user_hits['metadatas'][0][rank - 1]['source']}")
    print("-" * 60)
    print(doc[:350], "...")                                # preview first 350 chars
    print()

# ── Step 3: Build the grounded RAG prompt ────────────────────────────────────
user_context = "\n\n---\n\n".join(user_top_docs)           # join chunks with divider

user_rag_prompt = f"""You are a helpful pharmaceutical quality assistant.
Use ONLY the context provided below to answer the question.
If the answer is not found in the context, say "I cannot find that in the provided SOPs."

CONTEXT:
{user_context}

QUESTION: {user_question}

ANSWER:"""

# ── Step 4: Call HuggingFace / Qwen2.5 with the grounded prompt ───────────────
print("Sending to HuggingFace / Qwen2.5...\n")
user_response = hf_client.chat.completions.create(
    model="Qwen/Qwen2.5-72B-Instruct",
    messages=[{"role": "user", "content": user_rag_prompt}],
    max_tokens=512,
    temperature=0.7
)

# Extract and display the grounded answer
print("── Grounded Answer ───────────────────────────────────────────")
print(user_response.choices[0].message.content)

---

## 🤖 Section 5: AI Agents with LangGraph

> **Note:** This notebook is designed to run after the previous sections in `training_notebook.ipynb`.  
> It assumes the following objects are already in memory:
> - `hf_client` — HuggingFace Inference client for Qwen2.5-72B-Instruct (from nb_01)
> - `chroma_collection` — ChromaDB collection loaded with SOP documents (from nb_03)
> - `embedding_model` — sentence-transformers `all-MiniLM-L6-v2` (from nb_03)

> A `.env` file has been shared with you separately. Place it in the same folder as this notebook before running. It contains: `HF_TOKEN`  
> You can generate a free HF_TOKEN at: https://huggingface.co/settings/tokens  
> Select **'Read' access** — no payment required.

**Run the cells below first — we'll explain what just happened after.**

In [ ]:
# Install LangGraph, HuggingFace Hub client, and diagram dependencies
import subprocess
subprocess.run(
    ["pip", "install", "langgraph", "huggingface_hub", "networkx", "matplotlib", "--index-url", "https://pypi.org"],
    check=True
)
print("Packages ready.")

---
### Cell 1 — Define the Agent State and Tools

Every LangGraph agent has **State** — a shared memory object that every node reads from and writes to.  
Here our state is simply a growing list of messages, one entry per reasoning step.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
from typing import TypedDict, List, Dict, Any
import re

# LangGraph primitives
from langgraph.graph import StateGraph, END

# ── Agent State ───────────────────────────────────────────────────────────────
# TypedDict makes the shared state strongly typed and easy to inspect
class AgentState(TypedDict):
    messages: List[Dict[str, Any]]   # full conversation / reasoning trace
    user_question: str               # original question — never mutated
    tools_called: List[str]          # audit trail of which tools fired
    final_answer: str                # populated by the END node


# ── Tool 1: document_lookup ───────────────────────────────────────────────────
def document_lookup(question: str, state: AgentState) -> AgentState:
    """
    Queries the ChromaDB collection for the top-2 most relevant SOP chunks
    and appends the retrieved text to the agent's message history.
    """
    print("  [Tool] document_lookup called")
    print(f"  [Tool] Searching ChromaDB for: '{question}'")

    # Embed the user question with the same model used to build the index
    query_embedding = embedding_model.encode([question]).tolist()

    # Query ChromaDB — return the 2 closest chunks
    results = chroma_collection.query(
        query_embeddings=query_embedding,
        n_results=2,
        include=["documents", "metadatas"]
    )

    # Flatten returned documents into a readable string
    chunks = results["documents"][0]   # list of strings for first query
    metas  = results["metadatas"][0]   # parallel list of metadata dicts

    retrieved_text = ""
    for i, (chunk, meta) in enumerate(zip(chunks, metas), start=1):
        source = meta.get("source", "unknown document")
        retrieved_text += f"[Chunk {i} from '{source}']\n{chunk}\n\n"

    print(f"  [Tool] Retrieved {len(chunks)} chunk(s) from ChromaDB")

    # Record the tool result in the shared message history
    state["messages"].append({
        "role": "tool",
        "name": "document_lookup",
        "content": retrieved_text.strip()
    })
    state["tools_called"].append("document_lookup")
    return state


# ── Tool 2: deviation_classifier ─────────────────────────────────────────────
def deviation_classifier(description: str, state: AgentState) -> AgentState:
    """
    Classifies a deviation description as Critical / Major / Minor
    using keyword matching rules, without calling an LLM.
    """
    print("  [Tool] deviation_classifier called")
    print(f"  [Tool] Classifying: '{description[:80]}...'" if len(description) > 80 else f"  [Tool] Classifying: '{description}'")

    text_lower = description.lower()   # normalise for case-insensitive matching

    # Critical keywords — patient safety or product recall risk
    critical_keywords = ["contamination", "patient risk", "recall", "fatality", "death", "serious adverse"]

    # Major keywords — process integrity failures
    major_keywords = ["excursion", "out of spec", "oos", "process failure", "yield loss", "batch failure"]

    # Check Critical first (highest severity wins)
    if any(kw in text_lower for kw in critical_keywords):
        classification = "CRITICAL"
        rationale = f"Matched critical keyword in: '{description[:60]}'"

    # Then check Major
    elif any(kw in text_lower for kw in major_keywords):
        classification = "MAJOR"
        rationale = f"Matched major keyword in: '{description[:60]}'"

    # Default to Minor
    else:
        classification = "MINOR"
        rationale = "No critical or major keywords found — defaulting to Minor"

    result_text = f"Deviation Classification: **{classification}**\nRationale: {rationale}"
    print(f"  [Tool] Result → {classification}")

    # Append classification result to shared message history
    state["messages"].append({
        "role": "tool",
        "name": "deviation_classifier",
        "content": result_text
    })
    state["tools_called"].append("deviation_classifier")
    return state


print("✅ AgentState, document_lookup, and deviation_classifier defined.")

---
### Cell 2 — Define the Supervisor Node and Build the Graph

The **Supervisor** is the brain: it reads the current state, calls **Qwen2.5-72B-Instruct via the HuggingFace Inference API** to decide what to do next,  
then routes to the appropriate tool node — or directly to END if no tool is needed.

In [ ]:
# ── Supervisor Node ────────────────────────────────────────────────────────────
def supervisor(state: AgentState) -> AgentState:
    """
    Uses HuggingFace / Qwen2.5 to decide which tool (if any) to call next.
    Appends its reasoning to state['messages'].
    Sets state['next_node'] to control graph routing.
    """
    print("\n[Supervisor] Deciding next action...")

    question = state["user_question"]

    # Build a system prompt that instructs Qwen2.5 to act as a routing agent
    system_prompt = """You are a pharmaceutical quality supervisor agent.
You have access to two tools:
  1. document_lookup  — retrieves relevant SOP / policy chunks from a document store
  2. deviation_classifier — classifies a deviation as Minor, Major, or Critical

Given the user question below, decide which single tool to call first.
Reply with ONLY one of these exact strings (no punctuation, no explanation):
  document_lookup
  deviation_classifier
  END

Choose END only if the question has already been fully answered by a previous tool result.
"""

    # Summarise any tool results already in the message history
    tool_results_so_far = "\n".join(
        f"[{m['name']}]: {m['content']}"
        for m in state["messages"]
        if m["role"] == "tool"
    )

    # Full context handed to Qwen2.5
    user_message = f"User question: {question}"
    if tool_results_so_far:
        user_message += f"\n\nTool results so far:\n{tool_results_so_far}"

    # Call HuggingFace / Qwen2.5 — lightweight routing decision, low token budget
    response = hf_client.chat.completions.create(
        model="Qwen/Qwen2.5-72B-Instruct",
        messages=[
            {"role": "system",  "content": system_prompt},
            {"role": "user",    "content": user_message}
        ],
        max_tokens=20,        # routing answer is short — cap tokens to save cost
        temperature=0.0       # deterministic routing
    )

    # Extract the raw routing decision from Qwen2.5's response
    decision = response.choices[0].message.content.strip().lower()
    print(f"[Supervisor] Qwen2.5 routing decision: '{decision}'")

    # Normalise to valid node names (handle minor typos / extra whitespace)
    if "document" in decision:
        next_node = "document_lookup"
    elif "deviation" in decision or "classif" in decision:
        next_node = "deviation_classifier"
    else:
        next_node = "END"

    print(f"[Supervisor] Routing to → {next_node}")

    # Store the routing decision in state so the conditional edge can read it
    state["messages"].append({
        "role": "supervisor",
        "content": f"Decided to call: {next_node}"
    })
    state["next_node"] = next_node   # type: ignore[typeddict-unknown-key]
    return state


# ── Wrapper nodes — LangGraph node functions receive and return state ──────────
def node_document_lookup(state: AgentState) -> AgentState:
    """LangGraph node that calls document_lookup with the user question."""
    print("\n[Node] document_lookup")
    return document_lookup(state["user_question"], state)


def node_deviation_classifier(state: AgentState) -> AgentState:
    """LangGraph node that calls deviation_classifier with the user question."""
    print("\n[Node] deviation_classifier")
    return deviation_classifier(state["user_question"], state)


def node_end(state: AgentState) -> AgentState:
    """
    END node: asks HuggingFace / Qwen2.5 to synthesise all tool results into a
    final answer, then stores it in state['final_answer'].
    """
    print("\n[Node] END — generating final answer")

    question = state["user_question"]

    # Collect all tool outputs accumulated during this run
    tool_results = "\n\n".join(
        f"=== {m['name']} result ===\n{m['content']}"
        for m in state["messages"]
        if m["role"] == "tool"
    )

    synthesis_prompt = f"""You are a pharmaceutical quality assistant.
Using only the tool results below, write a clear, concise answer to the user's question.
Do not invent information beyond what the tools returned.

User question: {question}

Tool results:
{tool_results if tool_results else 'No tools were called.'}
"""

    # Final synthesis call to HuggingFace / Qwen2.5
    response = hf_client.chat.completions.create(
        model="Qwen/Qwen2.5-72B-Instruct",
        messages=[{"role": "user", "content": synthesis_prompt}],
        max_tokens=400,
        temperature=0.2
    )

    final = response.choices[0].message.content.strip()
    state["final_answer"] = final
    print("[Node] Final answer generated.")
    return state


# ── Routing function — reads next_node from state to direct graph flow ────────
def route_after_supervisor(state: AgentState) -> str:
    """Conditional edge: returns the name of the next node to visit."""
    return state.get("next_node", "END")   # type: ignore[typeddict-item]


# ── Build the StateGraph ──────────────────────────────────────────────────────
builder = StateGraph(AgentState)

# Register nodes — each is a Python function that accepts and returns AgentState
builder.add_node("supervisor",            supervisor)
builder.add_node("document_lookup",       node_document_lookup)
builder.add_node("deviation_classifier",  node_deviation_classifier)
builder.add_node("end_node",              node_end)

# Entry point — the graph always starts at the supervisor
builder.set_entry_point("supervisor")

# Conditional edge from supervisor → one of {document_lookup, deviation_classifier, end_node}
builder.add_conditional_edges(
    "supervisor",
    route_after_supervisor,
    {
        "document_lookup":      "document_lookup",
        "deviation_classifier": "deviation_classifier",
        "END":                  "end_node"
    }
)

# After each tool runs, return to supervisor so it can decide what to do next
builder.add_edge("document_lookup",      "supervisor")
builder.add_edge("deviation_classifier", "supervisor")

# end_node is terminal
builder.add_edge("end_node", END)

# Compile the graph into a runnable object
agent_graph = builder.compile()

print("✅ LangGraph agent compiled successfully.")

---
### Cell 3 — Run the Agent and Watch it Reason

We invoke the compiled graph with an initial state.  
Watch the console output — you can see exactly which node is visited and why.

In [ ]:
# ── Sample question — change this later in the "Try it yourself" cell ─────────
sample_question = "What is the SOP for handling an out-of-spec result in manufacturing?"

# ── Initial agent state ───────────────────────────────────────────────────────
initial_state: AgentState = {
    "messages":      [],             # empty — will be filled as nodes fire
    "user_question": sample_question,
    "tools_called":  [],             # audit trail starts empty
    "final_answer":  ""              # populated by end_node
}

print("=" * 65)
print(f"USER QUESTION: {sample_question}")
print("=" * 65)

# ── Invoke the graph ──────────────────────────────────────────────────────────
# stream_mode='values' gives us the state after each node fires
final_state = None
for step_state in agent_graph.stream(initial_state, stream_mode="values"):
    final_state = step_state   # keep overwriting — last one is the final state

print("\n" + "=" * 65)
print("Graph execution complete.")
print("=" * 65)

---
### Cell 4 — Print the Final Answer and Tool Audit Trail

In [ ]:
# ── Display results ────────────────────────────────────────────────────────────
print("\n" + "━" * 65)
print("AGENT FINAL ANSWER")
print("━" * 65)
print(final_state["final_answer"])

print("\n" + "━" * 65)
print("TOOLS CALLED (in order)")
print("━" * 65)
if final_state["tools_called"]:
    for i, tool_name in enumerate(final_state["tools_called"], start=1):
        print(f"  {i}. {tool_name}")
else:
    print("  (no tools called)")

print("\n" + "━" * 65)
print("FULL MESSAGE TRACE")
print("━" * 65)
for msg in final_state["messages"]:
    role    = msg.get("role", "?")
    name    = msg.get("name", "")         # tool name if present
    content = msg.get("content", "")[:200]  # truncate long tool outputs
    label   = f"{role}:{name}" if name else role
    print(f"  [{label}] {content}")

---
### Graph Diagram — Visualising the Agent Architecture

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Lilly brand colours ────────────────────────────────────────────────────────
LILLY_RED  = "#CC0000"   # Eli Lilly red for the Supervisor node
DARK_GREY  = "#404040"   # Dark grey for tool nodes
END_GREEN  = "#2E7D32"   # Green for the END node
EDGE_GREY  = "#888888"   # Grey for arrows

# ── Build a directed graph matching the LangGraph topology ───────────────────
G = nx.DiGraph()

# Add nodes
G.add_node("Supervisor")
G.add_node("document_lookup")
G.add_node("deviation_classifier")
G.add_node("END")

# Add directed edges (matches builder.add_edge calls above)
G.add_edge("Supervisor",           "document_lookup",      label="route")
G.add_edge("Supervisor",           "deviation_classifier", label="route")
G.add_edge("Supervisor",           "END",                  label="done")
G.add_edge("document_lookup",      "Supervisor",           label="return")
G.add_edge("deviation_classifier", "Supervisor",           label="return")

# ── Node positions (manual layout for clarity) ─────────────────────────────
pos = {
    "Supervisor":           (0.5,  0.7),
    "document_lookup":      (0.1,  0.3),
    "deviation_classifier": (0.9,  0.3),
    "END":                  (0.5,  0.0)
}

# ── Colour map — one colour per node ──────────────────────────────────────────
node_colours = {
    "Supervisor":           LILLY_RED,
    "document_lookup":      DARK_GREY,
    "deviation_classifier": DARK_GREY,
    "END":                  END_GREEN
}
colour_list = [node_colours[n] for n in G.nodes()]

# ── Draw ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_title("LangGraph Agent — Node & Edge Topology", fontsize=14, fontweight="bold", pad=20)

# Draw nodes
nx.draw_networkx_nodes(
    G, pos,
    node_color=colour_list,
    node_size=3000,
    ax=ax
)

# Draw node labels in white so they stand out on dark backgrounds
nx.draw_networkx_labels(
    G, pos,
    font_size=9,
    font_color="white",
    font_weight="bold",
    ax=ax
)

# Draw edges with slight curve so bidirectional arrows are both visible
nx.draw_networkx_edges(
    G, pos,
    edge_color=EDGE_GREY,
    arrows=True,
    arrowstyle="->",
    arrowsize=20,
    connectionstyle="arc3,rad=0.15",   # curved arrows — avoids overlap
    width=2,
    ax=ax
)

# Draw edge labels
edge_labels = nx.get_edge_attributes(G, "label")
nx.draw_networkx_edge_labels(
    G, pos,
    edge_labels=edge_labels,
    font_size=8,
    font_color="#333333",
    ax=ax
)

# ── Legend ────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(color=LILLY_RED,  label="Supervisor (LLM routing)"),
    mpatches.Patch(color=DARK_GREY,  label="Tool node"),
    mpatches.Patch(color=END_GREEN,  label="END (answer synthesis)"),
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=9)

ax.axis("off")   # hide axes — we only want the graph
plt.tight_layout()
plt.show()

---
## 📖 Theory: What Just Happened?

### State, Nodes, and Edges — in plain English

| Concept | What it is | Analogy |
|---------|-----------|--------|
| **State** | A shared dictionary every node reads and writes | A whiteboard in a meeting room — everyone can see it and add to it |
| **Node** | A Python function that does one job (call an LLM, run a tool, format output) | A person in the meeting with a specific role |
| **Edge** | An arrow that says "after Node A, go to Node B" | The agenda — who speaks after whom |
| **Conditional Edge** | An edge where the *next* node is decided at runtime | "If the data analyst's result is green, go to sign-off; if red, go back to investigation" |

---

### Why graph-based agents beat simple loops

A simple prompt loop looks like:  
`ask LLM → call tool → ask LLM → call tool → ...`  

The problem: **you can't see inside it.** When it goes wrong you have no idea which step failed, why it looped, or how to stop it.

LangGraph makes the structure **explicit**:

- **Visibility** — every node transition is logged. You watched each step print above.
- **Debuggability** — the full `messages` trace shows exactly what the LLM saw and decided at each point.
- **Human-in-the-loop** — you can insert a `human_review` node between any two steps. The graph pauses and waits for a person to approve before continuing. This is invaluable in regulated pharma workflows.
- **Controlled loops** — the supervisor can loop through tools multiple times, but the graph topology prevents infinite runaway because every path eventually reaches END.

> **Analogy:** A simple loop is a waterfall process with no checkpoints — it runs until it crashes. A LangGraph agent is a decision tree that can also *think* at each branch point. You get the flexibility of an LLM with the auditability of a flowchart.

---

### Why this matters in pharma / GxP contexts

- Audit trails map directly to graph edge logs  
- Human review steps are first-class citizens, not afterthoughts  
- Tool calls (e.g. LIMS queries, ERP lookups) are isolated nodes — easy to mock or replace  
- The graph *is* the SOP — you can generate a diagram and show it to a QA reviewer

---
## 🛠️ Try It Yourself

Change `user_question` below and re-run the cell.  
Watch how the Supervisor routes differently depending on what you ask:

- An SOP question → `document_lookup`  
- A deviation description → `deviation_classifier`  
- Try adding words like **"contamination"** or **"out of spec"** and see the classification change

**Tip:** Try asking about a *contamination* event and watch the classifier fire CRITICAL.

In [ ]:
# ── Change this question and re-run the cell ──────────────────────────────────
user_question = "A batch showed contamination during final QC inspection — how serious is this?"

# ── Fresh initial state for this run ─────────────────────────────────────────
run_state: AgentState = {
    "messages":      [],
    "user_question": user_question,
    "tools_called":  [],
    "final_answer":  ""
}

print("=" * 65)
print(f"USER QUESTION: {user_question}")
print("=" * 65)

# Run the graph and capture the final state
result_state = None
for step in agent_graph.stream(run_state, stream_mode="values"):
    result_state = step

# ── Print results ──────────────────────────────────────────────────────────────
print("\n" + "━" * 65)
print("FINAL ANSWER")
print("━" * 65)
print(result_state["final_answer"])

print("\n" + "━" * 65)
print(f"Tools called: {result_state['tools_called']}")
print("━" * 65)

---

> **Note:** This notebook is designed to run after the previous sections in `training_notebook.ipynb`. It assumes `hf_client`, `chroma_collection`, and `embedding_model` are already in memory.
>
> A `.env` file has been shared with you separately. Place it in the same folder as this notebook before running. It contains: `HF_TOKEN`  
> You can generate a free HF_TOKEN at: https://huggingface.co/settings/tokens  
> Select **'Read' access** — no payment required.

## 🔌 Section 6: MCP — Model Context Protocol

> **Run the cells below first — we'll explain what just happened after.**

---
### Step 1 — Build and Start the MCP Server

We'll implement a lightweight MCP-style server using **FastAPI** (served via `uvicorn` in a background thread).  
The server exposes two tools as HTTP endpoints and accepts a standard JSON envelope:  
```json
{ "tool": "tool_name", "args": { ... } }
```
This mirrors exactly how a real MCP server handles tool dispatch.

In [ ]:
# All packages for this section are pre-installed in the Codespace

import importlib

packages = ["fastapi", "uvicorn", "httpx", "huggingface_hub"]

all_good = True
for pkg in packages:
    try:
        importlib.import_module(pkg)
        print(f"✅ {pkg} is available")
    except ImportError:
        print(f"❌ {pkg} is missing — run: pip install {pkg}")
        all_good = False

if all_good:
    print("\n✅ All packages ready. Proceed to the next cell.")

In [ ]:
import threading        # Run the server in a background thread
import time             # Brief pause to let the server start
import json             # Pretty-print JSON at every step
import uvicorn          # ASGI server to serve FastAPI
from fastapi import FastAPI, Request    # Web framework for our MCP server
from fastapi.responses import JSONResponse

# ── Tool 1: get_sop_guidance ──────────────────────────────────────────────────
def get_sop_guidance(topic: str) -> str:
    """Return hardcoded SOP guidance text based on a keyword match."""
    topic_lower = topic.lower()  # Normalise to lowercase for matching

    if "temperature" in topic_lower:
        return "Immediately quarantine the batch and initiate a temperature excursion report per SOP-QA-042"
    elif "deviation" in topic_lower:
        return "Log the deviation in the LIMS system within 24 hours per SOP-QA-017"
    elif "capa" in topic_lower:
        return "Initiate CAPA form within 5 business days per SOP-QA-031"
    else:
        return "Please refer to the relevant SOP for guidance"


# ── Tool 2: classify_severity ─────────────────────────────────────────────────
def classify_severity(description: str) -> str:
    """Classify a deviation as Minor / Major / Critical using keyword rules."""
    desc_lower = description.lower()  # Normalise to lowercase for matching

    # Critical: patient safety or regulatory breach keywords
    if any(kw in desc_lower for kw in ["critical", "contamination", "patient", "recall", "sterility"]):
        return "Critical"
    # Major: significant process or quality impact
    elif any(kw in desc_lower for kw in ["major", "temperature excursion", "out of spec", "oos", "failure", "excursion"]):
        return "Major"
    # Minor: low-impact issues
    else:
        return "Minor"


# ── FastAPI app (our MCP server) ──────────────────────────────────────────────
app = FastAPI(title="MCP Tool Server")

@app.post("/call")  # Single endpoint that dispatches all tool calls
async def call_tool(request: Request):
    body = await request.json()          # Parse the incoming JSON envelope
    tool_name = body.get("tool")         # Which tool to invoke
    args      = body.get("args", {})     # Arguments for that tool

    # Route to the correct tool function
    if tool_name == "get_sop_guidance":
        result = get_sop_guidance(args.get("topic", ""))
    elif tool_name == "classify_severity":
        result = classify_severity(args.get("description", ""))
    else:
        # Unknown tool — return an error payload
        return JSONResponse({"error": f"Unknown tool: {tool_name}"}, status_code=404)

    return JSONResponse({"tool": tool_name, "result": result})


# ── Start the server in a daemon background thread ────────────────────────────
def run_server():
    """Target function for the background thread — starts uvicorn."""
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=8765,
        log_level="error"   # Suppress verbose uvicorn logs in the notebook
    )

# daemon=True means the thread dies automatically when the notebook kernel stops
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(2)  # Give uvicorn a moment to bind to the port before we call it
print("MCP Server running at http://localhost:8765")

---
### Step 2 — Call the MCP Server as a Client

Now we act as the **client** — sending raw HTTP POST requests with a JSON envelope.  
We'll print both the request and response at each step so the protocol is fully transparent.

In [ ]:
import httpx   # HTTP client for making requests to the MCP server

MCP_URL = "http://127.0.0.1:8765/call"  # The single tool-dispatch endpoint

# ── Call 1: get_sop_guidance ──────────────────────────────────────────────────
request_payload_1 = {
    "tool": "get_sop_guidance",          # Tool name — the MCP server routes on this
    "args": {"topic": "temperature"}      # Arguments specific to this tool
}

print("═" * 60)
print("TOOL CALL 1 — get_sop_guidance")
print("─" * 60)
print("REQUEST JSON:")
print(json.dumps(request_payload_1, indent=2))  # Show exactly what we're sending

response_1 = httpx.post(MCP_URL, json=request_payload_1)  # Fire the HTTP POST

print("\nRESPONSE JSON:")
print(json.dumps(response_1.json(), indent=2))  # Pretty-print the server's reply
print("HTTP Status:", response_1.status_code)

In [ ]:
# ── Call 2: classify_severity ─────────────────────────────────────────────────
request_payload_2 = {
    "tool": "classify_severity",                                      # Different tool
    "args": {"description": "temperature excursion of 3 degrees"}     # Different args
}

print("═" * 60)
print("TOOL CALL 2 — classify_severity")
print("─" * 60)
print("REQUEST JSON:")
print(json.dumps(request_payload_2, indent=2))  # Show the request envelope

response_2 = httpx.post(MCP_URL, json=request_payload_2)  # Fire the HTTP POST

print("\nRESPONSE JSON:")
print(json.dumps(response_2.json(), indent=2))  # Pretty-print the server's reply
print("HTTP Status:", response_2.status_code)

---
### Step 3 — Connect MCP to an LLM

Now we close the loop: **the LLM decides which MCP tool to call**, we execute the call against our server, and feed the result back to the LLM for a final answer.

This is the full agentic loop: `User → LLM → MCP Server → Tool → LLM → Final Answer`

In [ ]:
# hf_client is already initialised from nb_01 — we reference it here

# ── Define the two tools in HuggingFace / Qwen2.5's JSON-schema tool format ───
mcp_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_sop_guidance",
            "description": "Returns the relevant SOP action for a given topic keyword (e.g. temperature, deviation, capa)",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The topic or keyword to look up SOP guidance for"
                    }
                },
                "required": ["topic"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "classify_severity",
            "description": "Classifies the severity of a quality deviation as Minor, Major, or Critical",
            "parameters": {
                "type": "object",
                "properties": {
                    "description": {
                        "type": "string",
                        "description": "A plain-text description of the deviation or incident"
                    }
                },
                "required": ["description"]
            }
        }
    }
]

# ── Step A: Send the user question with tool definitions ──────────────────────
user_question = "We have a temperature excursion in batch XB-2024-001. What should we do and how severe is it?"

print("═" * 60)
print("STEP A — User Question")
print("─" * 60)
print(user_question)

# First LLM call: provide tools so the model can decide what to call
first_response = hf_client.chat.completions.create(
    model="Qwen/Qwen2.5-72B-Instruct",
    messages=[{"role": "user", "content": user_question}],
    tools=mcp_tools,              # Hand the tool schema to the LLM
    tool_choice="auto"            # Let the model decide if/which tools to call
)

first_message = first_response.choices[0].message  # Unwrap the response object

print("\nSTEP B — LLM Tool Choices")
print("─" * 60)
# Show every tool the model decided to invoke
for tc in (first_message.tool_calls or []):
    print(f"  Tool chosen : {tc.function.name}")
    print(f"  Arguments   : {tc.function.arguments}")

In [ ]:
# ── Step C: Execute each tool call via the MCP server ────────────────────────
# Build up the conversation history: original user message + LLM assistant turn
messages = [
    {"role": "user",      "content": user_question},
    first_message         # The assistant's turn containing its tool-call decisions
]

print("STEP C — Execute Tool Calls via MCP Server")
print("─" * 60)

tool_calls = first_message.tool_calls or []  # Safely handle if no tools were called

for tc in tool_calls:
    tool_name = tc.function.name                          # e.g. "get_sop_guidance"
    tool_args = json.loads(tc.function.arguments)         # Parse the JSON string of args
    tool_call_id = tc.id                                  # ID used to pair result to call

    # Build the MCP request envelope
    mcp_request = {"tool": tool_name, "args": tool_args}
    print(f"\n  MCP REQUEST  → {json.dumps(mcp_request, indent=4)}")

    # Send to the MCP server
    mcp_response = httpx.post(MCP_URL, json=mcp_request)
    tool_result  = mcp_response.json().get("result", "")
    print(f"  MCP RESPONSE ← {json.dumps(mcp_response.json(), indent=4)}")

    # Append the tool result to the message history in the expected format
    messages.append({
        "role":         "tool",
        "content":      tool_result,
        "tool_call_id": tool_call_id   # Must match the call ID so the model can pair them
    })

print("\n  All tool results collected — sending back to LLM...")

In [ ]:
# ── Step D: Send tool results back to HuggingFace / Qwen2.5 for the final answer
final_response = hf_client.chat.completions.create(
    model="Qwen/Qwen2.5-72B-Instruct",
    messages=messages    # Full history: user → LLM tool calls → tool results
)

final_answer = final_response.choices[0].message.content  # Extract the answer text

print("═" * 60)
print("STEP D — Final LLM Answer")
print("─" * 60)
print(final_answer)

---
### Flow Diagram — How MCP Connects Everything

In [ ]:
import matplotlib.pyplot as plt          # For drawing the architecture diagram
import matplotlib.patches as mpatches   # Rectangle shapes for the boxes
from matplotlib.patches import FancyArrowPatch  # Arrows between boxes

# ── Colour palette ─────────────────────────────────────────────────────────────
LILLY_RED  = "#C8102E"   # Lilly brand red — used for LLM boxes
DARK_GREY  = "#4A4A4A"   # Dark grey — used for MCP / Tool boxes
TEXT_WHITE = "white"     # Text colour inside dark boxes
ARROW_CLR  = "#333333"   # Arrow colour

fig, ax = plt.subplots(figsize=(14, 4))
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.axis("off")  # No axis ticks or borders needed

# ── Box definitions: (label, x_left, colour) ──────────────────────────────────
# Each box is 1.6 wide × 1.2 tall, centred vertically at y=1.4
boxes = [
    ("User",             0.3,  DARK_GREY),
    ("LLM\n(Qwen2.5)",   2.3,  LILLY_RED),
    ("MCP Server\n(FastAPI)", 4.3, DARK_GREY),
    ("Tool\nExecution",  6.3,  DARK_GREY),
    ("Response",         8.3,  DARK_GREY),
    ("LLM\n(Qwen2.5)",  10.3,  LILLY_RED),
    ("Final\nAnswer",   12.3,  DARK_GREY),
]

BOX_W, BOX_H, BOX_Y = 1.6, 1.2, 1.4  # Box dimensions and baseline y

for label, x, colour in boxes:
    # Draw the filled rectangle
    rect = mpatches.FancyBboxPatch(
        (x, BOX_Y), BOX_W, BOX_H,
        boxstyle="round,pad=0.1",
        linewidth=1.5, edgecolor="white",
        facecolor=colour
    )
    ax.add_patch(rect)
    # Label text centred inside the box
    ax.text(
        x + BOX_W / 2, BOX_Y + BOX_H / 2,
        label, color=TEXT_WHITE,
        ha="center", va="center",
        fontsize=9, fontweight="bold", linespacing=1.4
    )

# ── Arrows between boxes ───────────────────────────────────────────────────────
arrow_xs = [
    (0.3 + BOX_W, 2.3),          # User → LLM
    (2.3 + BOX_W, 4.3),          # LLM → MCP Server
    (4.3 + BOX_W, 6.3),          # MCP Server → Tool Execution
    (6.3 + BOX_W, 8.3),          # Tool Execution → Response
    (8.3 + BOX_W, 10.3),         # Response → LLM
    (10.3 + BOX_W, 12.3),        # LLM → Final Answer
]

ARROW_Y = BOX_Y + BOX_H / 2  # Vertical centre of boxes
for x_start, x_end in arrow_xs:
    ax.annotate(
        "",
        xy=(x_end, ARROW_Y), xytext=(x_start, ARROW_Y),
        arrowprops=dict(arrowstyle="->", color=ARROW_CLR, lw=1.8)
    )

ax.set_title("MCP Architecture — Full Agentic Loop", fontsize=13, fontweight="bold", pad=14)
plt.tight_layout()
plt.show()

---
### 🧠 What Just Happened? — MCP Explained

#### What is the Model Context Protocol (MCP)?

MCP is an **open standard for connecting AI models to external tools and data sources**. Think of it as a universal adapter — the same way USB-C replaced dozens of incompatible cables with one standard connector, MCP replaces the bespoke plumbing every team used to write to wire an LLM up to a database, API, or internal system.

Without MCP:  
```
LLM ──custom code──▶ Tool A
LLM ──custom code──▶ Tool B        (each connection is one-off, brittle, hard to share)
LLM ──custom code──▶ Tool C
```

With MCP:  
```
LLM ──MCP──▶ Any registered tool   (one standard, many tools, any model)
```

#### How does it work in practice?

| Step | Who acts | What happens |
|------|----------|--------------|
| 1 | **User** | Sends a natural-language question |
| 2 | **LLM** | Reads the tool schema and picks the right tool |
| 3 | **MCP Client** | Serialises the call into a standard JSON envelope |
| 4 | **MCP Server** | Receives the call, routes to the tool function, returns a result |
| 5 | **LLM** | Receives the result and synthesises a final answer |

#### Why does this matter for scalable agentic systems?

- **Decoupling**: The LLM doesn't need to know *how* a tool works — just its name, description, and input schema. You can swap or update tools without touching the model code.
- **Reusability**: Any MCP-compatible model can use any MCP server. Build a tool once; use it across GPT-4, Qwen2.5, Claude, and future models.
- **Auditability**: Because every tool call is a structured HTTP request/response, you get a natural audit trail — critical in regulated industries like pharma.
- **Security boundary**: The MCP server is the only component that touches internal systems. The LLM never has direct database or file system access.

---
### 🧪 Try It Yourself

Change the `topic` variable below to call different tools and see the MCP round-trip for yourself.  
Try: `"deviation"`, `"capa"`, `"sterility breach"`, or anything else.

In [ ]:
# ── Change this variable and re-run the cell ──────────────────────────────────
topic = "deviation"          # Try: "temperature", "capa", "sterility breach", "audit"

# Build the MCP request envelope
try_request = {"tool": "get_sop_guidance", "args": {"topic": topic}}

print("REQUEST:")
print(json.dumps(try_request, indent=2))    # Show exactly what we send

try_response = httpx.post(MCP_URL, json=try_request)   # Call the MCP server

print("\nRESPONSE:")
print(json.dumps(try_response.json(), indent=2))        # Show what comes back

# ── Also try classify_severity with a custom description ─────────────────────
description = "temperature excursion of 3 degrees found during batch release"

sev_request  = {"tool": "classify_severity", "args": {"description": description}}
sev_response = httpx.post(MCP_URL, json=sev_request)

print("\nSEVERITY REQUEST:")
print(json.dumps(sev_request, indent=2))
print("\nSEVERITY RESPONSE:")
print(json.dumps(sev_response.json(), indent=2))

---

> **Note:** This notebook is designed to run after the previous sections in `training_notebook.ipynb`.
>
> When merged and run top-to-bottom, the following objects are already in memory:
> - `hf_client` — HuggingFace Inference client for Qwen2.5-72B-Instruct *(created in Section 1)*
> - `chroma_collection` — ChromaDB collection loaded with SOP documents *(created in Section 4)*
> - `embedding_model` — sentence-transformers `all-MiniLM-L6-v2` *(created in Section 4)*
>
> A `.env` file has been shared with you separately. Place it in the same folder as this notebook before running. It contains: `HF_TOKEN`
> You can generate a free HF_TOKEN at: https://huggingface.co/settings/tokens — select **Read** access, no payment required.

## 🤝 Section 7: A2A — Agent to Agent Protocol

**Run the cells below first — we’ll explain what just happened after.**

In [ ]:
# Install packages required for this section
# networkx : graph data structure for the A2A architecture diagram
# matplotlib: rendering the diagram
import subprocess
subprocess.run(
    ['uv', 'pip', 'install', 'networkx', 'matplotlib', '--default-index', 'https://pypi.org'],
    check=True
)
print('Packages ready.')

In [ ]:
# ── Cell 1: DocumentAgent ─────────────────────────────────────────────────────
# Retrieves the most relevant SOP chunks from ChromaDB for a given task.
# Does NOT call an LLM — purely deterministic vector retrieval.

class DocumentAgent:
    """Specialist agent: semantic SOP retrieval via ChromaDB."""

    name = 'DocumentAgent'   # identifier used in message dicts and print output

    def __init__(self):
        # Always print on init so instantiation is visible in the output
        print(f'[{self.name}] Initialised — ready to query ChromaDB.')

    def run(self, task: str) -> str:
        """Embed the task and retrieve the top-2 SOP chunks from ChromaDB."""

        # ── Entry print: show agent name and the task it received ─────────────
        print(f'\n[{self.name}] run() called.')
        print(f'[{self.name}] Task received: "{task}"')

        # Embed the incoming task with the shared sentence-transformer model
        # embedding_model lives in memory from nb_03_rag_pipeline.ipynb
        query_embedding = embedding_model.encode([task]).tolist()  # shape (1, 384) → list

        # Query ChromaDB for the 2 most semantically similar SOP chunks
        results = chroma_collection.query(
            query_embeddings=query_embedding,
            n_results=2,                          # retrieve the two closest chunks
            include=['documents', 'metadatas']    # return text + source metadata
        )

        # ChromaDB wraps every result in an outer list (one entry per query)
        chunks = results['documents'][0]   # list of retrieved document strings
        metas  = results['metadatas'][0]   # parallel list of metadata dicts

        # Build a labelled, readable result string from the retrieved chunks
        retrieved_text = ''
        for i, (chunk, meta) in enumerate(zip(chunks, metas), start=1):
            source = meta.get('source', 'unknown document')   # human-readable source label
            retrieved_text += f'[Chunk {i} — {source}]\n{chunk}\n\n'  # labelled block

        # Provide a fallback message if the collection returned nothing
        result = retrieved_text.strip() if retrieved_text.strip() else 'No relevant documents found.'

        # ── Exit print: show agent name, summary, and result preview ──────────
        print(f'[{self.name}] Retrieved {len(chunks)} SOP chunk(s) from ChromaDB.')
        print(f'[{self.name}] Result returned (preview): "{result[:140]}..."')

        return result   # full retrieved text handed back to the Orchestrator


# ── Smoke-test: instantiate to confirm no import or runtime errors ─────────────
_test_doc = DocumentAgent()
print('\nCell 1 complete — DocumentAgent defined successfully.')

In [ ]:
# ── Cell 2: ClassifierAgent ───────────────────────────────────────────────────
# Classifies a deviation description as Minor / Major / Critical.
# Uses keyword matching rules — fully deterministic, zero LLM calls.

class ClassifierAgent:
    """Specialist agent: keyword-rule severity classifier (Minor/Major/Critical)."""

    name = 'ClassifierAgent'   # identifier used in message dicts and print output

    # ── Keyword rule sets — class-level so they are visible without instantiation ──
    CRITICAL_KEYWORDS = [
        'contamination', 'patient risk', 'recall', 'fatality',
        'death', 'serious adverse', 'sterility breach'
    ]
    MAJOR_KEYWORDS = [
        'temperature excursion', 'excursion', 'out of spec',
        'oos', 'process failure', 'yield loss', 'batch failure'
    ]

    def __init__(self):
        # Report how many rules were loaded so the rule set is visible
        print(
            f'[{self.name}] Initialised — '
            f'{len(self.CRITICAL_KEYWORDS)} critical keywords, '
            f'{len(self.MAJOR_KEYWORDS)} major keywords loaded.'
        )

    def run(self, task: str) -> str:
        """Match keywords in task and return a severity classification with rationale."""

        # ── Entry print: show agent name and the task it received ─────────────
        print(f'\n[{self.name}] run() called.')
        print(f'[{self.name}] Task received: "{task}"')

        text_lower = task.lower()   # normalise once for all comparisons

        # Find the first matching keyword at each severity level
        matched_critical = next(
            (kw for kw in self.CRITICAL_KEYWORDS if kw in text_lower), None
        )
        matched_major = next(
            (kw for kw in self.MAJOR_KEYWORDS if kw in text_lower), None
        )

        # Apply severity priority: Critical > Major > Minor (highest wins)
        if matched_critical:
            classification = 'CRITICAL'
            rationale      = f"Matched critical keyword: '{matched_critical}'"
        elif matched_major:
            classification = 'MAJOR'
            rationale      = f"Matched major keyword: '{matched_major}'"
        else:
            classification = 'MINOR'
            rationale      = 'No critical or major keywords detected — default classification.'

        # Compose the result string returned to the Orchestrator
        result = f'Severity: {classification}\nRationale: {rationale}'

        # ── Exit print: show agent name, classification, and full result ──────
        print(f'[{self.name}] Classification → {classification}')
        print(f'[{self.name}] Result returned: "{result}"')

        return result   # severity + rationale handed back to the Orchestrator


# ── Smoke-test: instantiate to confirm no errors ──────────────────────────────
_test_clf = ClassifierAgent()
print('\nCell 2 complete — ClassifierAgent defined successfully.')

In [ ]:
# ── Cell 3: OrchestratorAgent ─────────────────────────────────────────────────
# The central hub of the A2A system.
# Makes TWO calls to HuggingFace / Qwen2.5:
#   1. Routing call  — decides which specialist agent(s) to involve
#   2. Synthesis call — composes the final answer from agent results
# All inter-agent messages are stored as dicts and printed for full visibility.

class OrchestratorAgent:
    """Orchestrator: routes tasks to specialists and synthesises the final answer."""

    name = 'OrchestratorAgent'   # identifier used in message dicts and print output

    def __init__(self, document_agent, classifier_agent):
        self.document_agent   = document_agent    # specialist for SOP retrieval
        self.classifier_agent = classifier_agent  # specialist for severity classification
        self.message_log      = []                # list of inter-agent message dicts

        print(
            f'[{self.name}] Initialised with specialists: '
            f'{document_agent.name}, {classifier_agent.name}.'
        )

    # ── Private helper: routing decision via Qwen2.5 ──────────────────────────
    def _decide_agents(self, question: str) -> list:
        """Ask Qwen2.5 which specialist agent(s) to involve for this question."""

        # System prompt instructs the model to reply with exactly one routing token
        routing_system_prompt = (
            'You are an orchestrator for a pharmaceutical quality system.\n'
            'You have two specialist agents available:\n'
            '  - document_agent   : retrieves relevant SOP / policy chunks from a document store\n'
            '  - classifier_agent : classifies a deviation as Minor, Major, or Critical\n'
            '\n'
            'Given the user question below, decide which agent(s) to involve.\n'
            'Reply with ONLY one of these exact options (no explanation, no punctuation):\n'
            '  document_agent\n'
            '  classifier_agent\n'
            '  both\n'
            '\n'
            "Choose 'both' if the question asks about BOTH what the SOP says AND the severity/seriousness of the issue."
        )

        # Short routing call — temperature 0 for determinism, max_tokens 10 (one word)
        # hf_client lives in memory from nb_01_setup_tokenisation.ipynb
        routing_response = hf_client.chat.completions.create(
            model='Qwen/Qwen2.5-72B-Instruct',
            messages=[
                {'role': 'system', 'content': routing_system_prompt},
                {'role': 'user',   'content': f'Question: {question}'}
            ],
            max_tokens=10,    # routing answer is a single word — cap tokens tightly
            temperature=0.0   # deterministic — same question always routes the same way
        )

        decision = routing_response.choices[0].message.content.strip().lower()
        print(f'[{self.name}] Routing decision from Qwen2.5: "{decision}"')

        # Map the text decision to a list of agent instances
        if 'both' in decision:
            return [self.document_agent, self.classifier_agent]  # involve both specialists
        elif 'classifier' in decision:
            return [self.classifier_agent]                        # severity question only
        else:
            return [self.document_agent]                          # SOP/policy question only

    # ── Private helper: create and log an inter-agent message dict ─────────────
    def _record_message(self, sender: str, task: str, result: str) -> dict:
        """Build a message dict, append it to the log, and print it."""

        # Standard A2A message envelope: who sent it, what they were asked, what they returned
        message = {
            'sender': sender,   # name of the agent that produced this result
            'task':   task,     # the task string that was delegated
            'result': result    # the string result the agent returned
        }
        self.message_log.append(message)   # accumulate in the full log

        # Print the message dict so inter-agent communication is fully visible
        print(f'\n\ud83d\udce8 Message recorded:')
        print(f'   sender : {message["sender"]}')
        print(f'   task   : {message["task"][:80]}{"..." if len(message["task"]) > 80 else ""}')
        print(f'   result : {message["result"][:140]}{"..." if len(message["result"]) > 140 else ""}')

        return message

    # ── Public entry point ────────────────────────────────────────────────────
    def run(self, question: str) -> str:
        """Receive a question, delegate to agents, collect results, synthesise answer."""

        DIVIDER = '=' * 65   # reusable visual divider

        print(f'\n{DIVIDER}')
        print(f'[{self.name}] run() called.')
        print(f'[{self.name}] Received question: "{question}"')
        print(DIVIDER)

        # ── Step 1: ask Qwen2.5 which specialists to involve ──────────────────
        agents_to_use = self._decide_agents(question)
        agent_names   = [a.name for a in agents_to_use]
        print(f'[{self.name}] Agents selected: {agent_names}')

        agent_results = {}   # results keyed by agent name, used for synthesis

        # ── Step 2: delegate to each chosen agent in order ────────────────────
        for agent in agents_to_use:
            print(f'\n[{self.name}] Delegating to {agent.name}...')

            result = agent.run(question)   # call the specialist agent

            # Confirm the result was received back
            print(f'[{agent.name}] Result returned to {self.name}.')

            # Record and print the inter-agent message dict
            self._record_message(
                sender=agent.name,
                task=question,
                result=result
            )

            agent_results[agent.name] = result   # store for the synthesis step

        # ── Step 3: synthesise final answer via Qwen2.5 ───────────────────────
        print(f'\n[{self.name}] Synthesising final answer from {len(agent_results)} agent result(s)...')

        # Build a structured context block from all collected agent outputs
        context_parts = []
        if DocumentAgent.name in agent_results:
            context_parts.append(
                '=== SOP Documentation (from DocumentAgent) ===\n'
                + agent_results[DocumentAgent.name]
            )
        if ClassifierAgent.name in agent_results:
            context_parts.append(
                '=== Severity Classification (from ClassifierAgent) ===\n'
                + agent_results[ClassifierAgent.name]
            )

        context = '\n\n'.join(context_parts)   # join context sections with blank line

        synthesis_prompt = (
            'You are a pharmaceutical quality assistant.\n'
            'Using ONLY the specialist agent outputs below, write a clear and concise answer '
            'to the user question. Do not invent information beyond what the agents returned.\n\n'
            + context
            + f'\n\nUser question: {question}\n'
            'Provide a well-structured answer that directly addresses all parts of the question.'
        )

        # Synthesis call — slightly more creative than routing (temperature 0.3)
        synthesis_response = hf_client.chat.completions.create(
            model='Qwen/Qwen2.5-72B-Instruct',
            messages=[{'role': 'user', 'content': synthesis_prompt}],
            max_tokens=512,
            temperature=0.3   # slightly creative for natural language output
        )

        final_answer = synthesis_response.choices[0].message.content.strip()

        print(f'[{self.name}] Final answer synthesised.')
        return final_answer   # returned to the caller (Cell 4 or Try it yourself)


# ── Smoke-test: instantiate with the agents from previous cells ───────────────
_test_orch = OrchestratorAgent(
    document_agent   = _test_doc,
    classifier_agent = _test_clf
)
print('\nCell 3 complete — OrchestratorAgent defined successfully.')

In [ ]:
# ── Cell 4: Full A2A Demo ─────────────────────────────────────────────────────
# Instantiates all three agents and runs the complete A2A flow.
# Watch the console: every delegation, message dict, and synthesis step is printed.

DIVIDER = '=' * 65   # visual divider reused throughout this cell

# ── Instantiate fresh agents for the demo ─────────────────────────────────────
print('Initialising agents...\n')
doc_agent        = DocumentAgent()    # specialist: SOP retrieval via ChromaDB
classifier_agent = ClassifierAgent()  # specialist: keyword-rule severity classification
orchestrator     = OrchestratorAgent( # hub: routing + synthesis via Qwen2.5
    document_agent   = doc_agent,
    classifier_agent = classifier_agent
)

# ── The demo question ─────────────────────────────────────────────────────────
# Deliberately asks TWO things: what the SOP says AND how severe the issue is.
# This should trigger the Orchestrator to involve BOTH specialist agents.
demo_question = (
    'We had a temperature excursion of 3\u00b0C for 2 hours in a manufacturing batch. '
    'What does our SOP say and how severe is this?'
)

print(f'\n{DIVIDER}')
print(f'Orchestrator received question: {demo_question}')
print(DIVIDER)

# ── Run the full A2A flow ─────────────────────────────────────────────────────
# The orchestrator will:
#   1. Ask Qwen2.5 which agents to involve  → prints routing decision
#   2. Delegate to DocumentAgent            → prints delegation + result
#   3. Delegate to ClassifierAgent          → prints delegation + result
#   4. Record inter-agent message dicts     → prints each message
#   5. Synthesise via Qwen2.5               → prints synthesis step
print('\nOrchestrator delegating to DocumentAgent...')
print('Orchestrator delegating to ClassifierAgent...')
print('Orchestrator synthesising final answer...')
print('(All of the above happen inside orchestrator.run() below — watch the output)\n')

final_answer = orchestrator.run(demo_question)   # single call drives the whole A2A flow

# ── Print the final synthesised answer ───────────────────────────────────────
print(f'\n{DIVIDER}')
print('FINAL ANSWER')
print(DIVIDER)
print(final_answer)

# ── Print the full inter-agent message log ────────────────────────────────────
print(f'\n{DIVIDER}')
print('INTER-AGENT MESSAGE LOG (all messages exchanged this run)')
print(DIVIDER)
for i, msg in enumerate(orchestrator.message_log, start=1):
    print(f'\n  Message {i}:')
    print(f'    sender : {msg["sender"]}')
    print(f'    task   : {msg["task"][:80]}...')
    print(f'    result : {msg["result"][:140]}...')

In [ ]:
# ── A2A Architecture Diagram ──────────────────────────────────────────────────
# Directed graph showing the full message flow:
#   User → Orchestrator → [DocumentAgent, ClassifierAgent] → Orchestrator → Final Answer

import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Colour palette ─────────────────────────────────────────────────────────────
LILLY_RED    = '#C8102E'   # Eli Lilly red — Orchestrator (LLM-powered hub)
DARK_GREY    = '#404040'   # Dark grey — specialist agents (deterministic)
USER_BLUE    = '#1565C0'   # Blue — User (entry point)
ANSWER_GREEN = '#2E7D32'   # Green — Final Answer (exit point)
EDGE_GREY    = '#666666'   # Subtle grey for arrows

# ── Build the directed graph ───────────────────────────────────────────────────
G = nx.DiGraph()

# Register all five nodes
G.add_nodes_from(['User', 'Orchestrator', 'DocumentAgent', 'ClassifierAgent', 'Final Answer'])

# Each edge: (source, target, label) — label shown on the arrow
edge_data = [
    ('User',            'Orchestrator',    'question'),
    ('Orchestrator',    'DocumentAgent',   'delegates'),
    ('Orchestrator',    'ClassifierAgent', 'delegates'),
    ('DocumentAgent',   'Orchestrator',    'returns result'),
    ('ClassifierAgent', 'Orchestrator',    'returns result'),
    ('Orchestrator',    'Final Answer',    'synthesises'),
]
for src, dst, lbl in edge_data:
    G.add_edge(src, dst, label=lbl)   # store label as edge attribute for drawing

# ── Manual node positions: hierarchical layout for clarity ────────────────────
pos = {
    'User':            (0.50,  1.00),   # top centre — entry point
    'Orchestrator':    (0.50,  0.58),   # middle centre — hub
    'DocumentAgent':   (0.15,  0.18),   # bottom left — specialist
    'ClassifierAgent': (0.85,  0.18),   # bottom right — specialist
    'Final Answer':    (0.50, -0.12),   # bottom centre — exit point
}

# ── Colour list ordered to match G.nodes() iteration ──────────────────────────
node_colour_map = {
    'User':            USER_BLUE,
    'Orchestrator':    LILLY_RED,
    'DocumentAgent':   DARK_GREY,
    'ClassifierAgent': DARK_GREY,
    'Final Answer':    ANSWER_GREEN,
}
colour_list = [node_colour_map[n] for n in G.nodes()]   # must match node iteration order

# ── Draw ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_title(
    'A2A Architecture — Agent to Agent Communication Flow',
    fontsize=14, fontweight='bold', color=LILLY_RED, pad=22
)

# Draw nodes as large circles so labels fit comfortably
nx.draw_networkx_nodes(
    G, pos,
    node_color=colour_list,
    node_size=3800,
    ax=ax
)

# White bold labels on dark backgrounds for readability
nx.draw_networkx_labels(
    G, pos,
    font_size=9,
    font_color='white',
    font_weight='bold',
    ax=ax
)

# Curved arrows so bidirectional pairs (delegate / return) are both visible
nx.draw_networkx_edges(
    G, pos,
    edge_color=EDGE_GREY,
    arrows=True,
    arrowstyle='->',
    arrowsize=20,
    connectionstyle='arc3,rad=0.15',   # curve prevents overlap of bidirectional arrows
    width=2,
    ax=ax
)

# Edge labels from the 'label' attribute stored during graph construction
edge_labels = nx.get_edge_attributes(G, 'label')
nx.draw_networkx_edge_labels(
    G, pos,
    edge_labels=edge_labels,
    font_size=8,
    font_color='#222222',
    ax=ax
)

# ── Legend ────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(color=USER_BLUE,    label='User (entry point)'),
    mpatches.Patch(color=LILLY_RED,    label='Orchestrator (LLM-powered routing + synthesis)'),
    mpatches.Patch(color=DARK_GREY,    label='Specialist Agent (deterministic, no LLM)'),
    mpatches.Patch(color=ANSWER_GREEN, label='Final Answer (exit point)'),
]
ax.legend(handles=legend_handles, loc='upper right', fontsize=9)

ax.axis('off')    # hide axes — only the graph matters
plt.tight_layout()
plt.show()

---
## 📖 Theory: A2A — Agent to Agent Protocol

### What is A2A?

**Agent to Agent (A2A)** is a design pattern where multiple AI agents collaborate by passing structured messages to each other. Instead of one monolithic prompt trying to handle everything, you build a **network of specialists** — each with a single, well-defined responsibility — coordinated by an **orchestrator** that decides who does what, collects the results, and composes the final output.

In the demo above:
- The **Orchestrator** asked Qwen2.5: *"which agents should I involve for this question?"*
- It delegated to the **DocumentAgent** (pure vector retrieval — no LLM)
- It delegated to the **ClassifierAgent** (pure keyword rules — no LLM)
- It collected both results and asked Qwen2.5 to synthesise a final answer

Only **two LLM calls** were made. The two specialist agents ran entirely without touching a model.

---

### Why Modular Agents Beat Monolithic Prompts

| | Single Monolithic Prompt | Multi-Agent A2A |
|---|---|---|
| **Reliability** | One prompt fails → everything fails | Each agent can be tested and validated independently |
| **Auditability** | Hard to trace which part of the prompt drove the answer | The message log shows exactly which agent contributed what |
| **Maintainability** | Changing one capability risks breaking others | Swap or update one agent without touching the rest |
| **Scalability** | Prompt grows unboundedly as you add capabilities | Add a new specialist without changing existing ones |
| **Cost** | Every call burns the full prompt context | Specialists can skip the LLM entirely (retrieval, rules) |

---

### When to Use Multi-Agent vs Single-Agent

**Use a single agent when:**
- The task is narrow, well-defined, and self-contained
- Latency matters more than modularity
- You are prototyping and iteration speed is the priority

**Use multi-agent A2A when:**
- The task requires different *types* of intelligence (retrieval + reasoning + classification)
- You need auditability — stakeholders need to see which component drove each part of the answer
- You are building something that will be maintained and extended over time
- Different agents need different data access or permission levels
- Some subtasks are deterministic (rules, lookups) and should never touch an LLM

---

### How This Pattern Scales to Enterprise Workflows

In a real pharmaceutical quality system, the same pattern grows naturally:

```
                    ┌───────────────────────────────────────┐
                    │           Orchestrator                │
                    │  (routes, collects, synthesises)      │
                    └───────────────────────────────────────┘
                   /        |          |           \
            ┌──────┐  ┌──────┐  ┌──────────┐  ┌───────────┐
            │ SOP  │  │ LIMS │  │ Deviation │  │ Regulatory│
            │Agent │  │Agent │  │ Classifier│  │   Agent   │
            │(RAG) │  │(API) │  │  (rules)  │  │  (RAG)   │
            └──────┘  └──────┘  └──────────┘  └───────────┘
```

Each specialist can be:
- A **retrieval agent** — vector DB query (as in this demo)
- An **API agent** — LIMS, ERP, TrackWise call
- A **rules-based agent** — deterministic classification (as in this demo)
- Another **LLM agent** — for complex sub-reasoning tasks
- A **human-in-the-loop checkpoint** — pause for regulatory approval before continuing

The Orchestrator stays simple: **route → collect → synthesise**. Complexity lives in the specialists, not in one sprawling prompt.

> **The key insight for regulated environments:** A2A gives you a structured message log that maps directly to an audit trail — *which agent said what, based on which input, at which step.* In pharmaceutical QA, this traceability is not optional.

In [ ]:
# ── Try It Yourself ───────────────────────────────────────────────────────────
# Change user_question below and re-run this cell.
# Watch which agents the Orchestrator chooses to involve — and why.
#
# Suggestions to try:
#   'What is the SOP for handling an out-of-specification laboratory result?'
#     → expect: document_agent only  (SOP question, no severity asked)
#
#   'We discovered contamination in Bioreactor Suite 4 during batch release.'
#     → expect: classifier_agent only  (severity question, 'contamination' → CRITICAL)
#
#   'Temperature excursion in cold store — what does the SOP say and how bad is it?'
#     → expect: both  (asks for SOP guidance AND severity)

user_question = 'We found a contamination event during final QC inspection. What should we do and how serious is this?'
# ↑ Change this line, then re-run the cell

DIVIDER = '=' * 65   # visual divider

# ── Instantiate fresh agents for this run ─────────────────────────────────────
try_doc        = DocumentAgent()
try_clf        = ClassifierAgent()
try_orch       = OrchestratorAgent(
    document_agent   = try_doc,
    classifier_agent = try_clf
)

print(f'\n{DIVIDER}')
print(f'YOUR QUESTION: {user_question}')
print(DIVIDER)

# ── Run the full A2A flow ─────────────────────────────────────────────────────
try_answer = try_orch.run(user_question)

# ── Display the final answer ──────────────────────────────────────────────────
print(f'\n{DIVIDER}')
print('FINAL ANSWER')
print(DIVIDER)
print(try_answer)

# ── Show which agents were involved this run ──────────────────────────────────
print(f'\n{DIVIDER}')
print('AGENTS INVOLVED THIS RUN')
print(DIVIDER)
agents_used = list({msg['sender'] for msg in try_orch.message_log})   # unique senders
for agent_name in sorted(agents_used):   # sort for consistent display order
    print(f'  \u2713 {agent_name}')
print(f'\nTotal inter-agent messages exchanged: {len(try_orch.message_log)}')